# Visual ML Foundations: Self-Supervised Contrastive Learning (SimCLR)
### Training Encoders on Millions of Unlabeled Medical Images Without Labels

---

## 1. Concept Overview
Labeling medical images is prohibitively expensive ($50+ per scan for radiologist annotation).
**Self-Supervised Contrastive Learning (SimCLR / MoCo / DINO)** trains models on unannotated images:
1. Take an unlabeled MRI slice $x$.
2. Apply two different random stochastic augmentations to generate views $x_i$ and $x_j$ (positive pair).
3. The model minimizes **NT-Xent Loss (Normalized Temperature-scaled Cross Entropy)**:
   $$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k} \exp(\text{sim}(z_i, z_k) / \tau)}$$
4. Pulls representations of the same image together, while pushing all other images in the batch apart!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import rotate, gaussian_filter

# Original unannotated knee MRI
np.random.seed(42)
y, x = np.ogrid[:100, :100]
x_orig = np.zeros((100, 100))
x_orig[((y - 40)**2 + (x - 50)**2) < 22**2] = 0.8
x_orig[((y - 75)**2 + (x - 50)**2) < 20**2] = 0.7

# Generate Positive Pair (Augmented Views of SAME image)
view_i = rotate(x_orig, angle=6, reshape=False) + np.random.normal(0, 0.05, (100, 100))
view_j = gaussian_filter(x_orig, sigma=1.0) * 1.2

# Generate Negative Sample (DIFFERENT image)
x_neg = np.zeros((100, 100))
x_neg[((y - 50)**2 + (x - 50)**2) < 35**2] = 0.65

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(x_orig, cmap='bone')
axes[0].set_title("Source Unlabeled Image x", fontsize=11, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(view_i, cmap='bone')
axes[1].set_title("Positive View x_i (PULL)", fontsize=11, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(view_j, cmap='bone')
axes[2].set_title("Positive View x_j (PULL)", fontsize=11, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(x_neg, cmap='bone')
axes[3].set_title("Negative Sample x_neg (PUSH)", fontsize=11, fontweight='bold')
axes[3].axis('off')

plt.suptitle("Self-Supervised Contrastive Representation Learning Dynamics", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Contrastive positive/negative pair dynamics visualized.")
